<a href="https://www.kaggle.com/code/pcsciprav/chess-bot-fun?scriptVersionId=356973301" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
%%bash
sudo apt-get update
sudo apt-get -y install git python3-pip gcc-10 g++-10 zlib1g zlib1g-dev ninja-build
pip3 install meson --break-system-packages

In [ ]:
%%bash
cd /kaggle/working
#rm -rf stockfish* lc0* match_results.pgn
# 3. Download the actual, fully-compiled STOCKFISH 19 Linux binary
wget https://github.com/official-stockfish/Stockfish/releases/download/sf_19/stockfish-linux-x86-64-universal.tar.gz
tar -xf stockfish-linux-x86-64-universal.tar.gz
chmod +x /kaggle/working/stockfish/stockfish-linux-x86-64-universal

In [ ]:
%%bash
git clone --depth 1 https://github.com/LeelaChessZero/lc0.git /kaggle/working/lc0
cd /kaggle/working/lc0
git submodule update --init --recursive
wget https://storage.lczero.org/files/networks-contrib/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz
export PATH=/usr/local/cuda/bin:$PATH
export LD_LIBRARY_PATH=/usr/local/cuda/lib64:$LD_LIBRARY_PATH

In [ ]:
!/kaggle/working/lc0/build.sh release

In [ ]:
%%bash
git clone --depth 1 https://github.com/lucasart/c-chess-cli.git /kaggle/working/c-chess-cli
cd /kaggle/working/c-chess-cli
python3 make.py

In [1]:
!chmod +x /kaggle/working/lc0/build/release/lc0

In [4]:
%%bash

cat > /kaggle/working/lc0_gpu0.sh <<'EOF'
#!/bin/bash
export CUDA_VISIBLE_DEVICES=0
exec /kaggle/working/lc0/build/release/lc0 "$@"
EOF

cat > /kaggle/working/lc0_gpu1.sh <<'EOF'
#!/bin/bash
export CUDA_VISIBLE_DEVICES=1
exec /kaggle/working/lc0/build/release/lc0 "$@"
EOF
cat << 'EOF' > process1.sh 

chmod +x /kaggle/working/lc0_gpu0.sh /kaggle/working/lc0_gpu1.sh

RUY_LOPEZ_FEN="rn1q2nr/ppp1kppp/B6B/3pp3/3PP3/b6b/PPP1KPPP/RN1Q2NR w - - 4 6"
echo "${RUY_LOPEZ_FEN}" > /kaggle/working/ruy_lopez_start.epd

/kaggle/working/c-chess-cli/c-chess-cli \
  -each tc=50/1 \
  -engine name=Lc0_toolow cmd=/kaggle/working/lc0_gpu0.sh \
    option.Backend=cuda-fp16 \
    "option.WeightsFile=/kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz" \
    option.Cpuct=0.01 \
    option.PolicyTemperature=0.00005 \
    option.SmartPruningFactor=0.25 \
    option.CPuctFactor=0.1 \
  -engine name=Lc0_superlow cmd=/kaggle/working/lc0_gpu0.sh \
    option.Backend=cuda-fp16 \
    "option.WeightsFile=/kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz" \
    option.Cpuct=0 \
    option.PolicyTemperature=0.04 \
    option.SmartPruningFactor=5.0 \
    option.CPuctFactor=0.4 \
  -openings file=/kaggle/working/ruy_lopez_start.epd \
  -repeat \
  -rounds 200 \
  -pgn /kaggle/working/lc0basetrain.pgn
EOF
cat << 'EOF' > process2.sh 
chmod +x /kaggle/working/lc0_gpu0.sh /kaggle/working/lc0_gpu1.sh

RUY_LOPEZ_FEN="rnb1kbnr/ppp1pppp/3q4/3p4/3P4/3Q4/PPP1PPPP/RNB1KBNR w KQkq - 2 3"
echo "${RUY_LOPEZ_FEN}" > /kaggle/working/ruy_lopez_start.epd

/kaggle/working/c-chess-cli/c-chess-cli \
  -each tc=20/1 \
  -engine name=Lc0_maxtemp cmd=/kaggle/working/lc0_gpu1.sh \
    option.Backend=cuda-fp16 \
    "option.WeightsFile=/kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz" \
    option.Cpuct=1904.4 \
    option.PolicyTemperature=240.00214 \
    option.SmartPruningFactor=5.0 \
    option.CPuctFactor=0.4 \
-engine name=Lc0_hightemp cmd=/kaggle/working/lc0_gpu1.sh \
    option.Backend=cuda-fp16 \
    "option.WeightsFile=/kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz" \
    option.Cpuct=20 \
    option.PolicyTemperature=4 \
    option.SmartPruningFactor=1.0 \
    option.CPuctFactor=1 \
  -openings file=/kaggle/working/ruy_lopez_start.epd \
  -repeat \
  -rounds 200 \
  -pgn /kaggle/working/lc0basetrain.pgn
EOF

In [2]:
!chmod +x /kaggle/working/lc0/build/release/lc0

In [6]:
%%bash
bash process1.sh &
bash process2.sh &
wait

[1] Started game 1 of 200 (Lc0_hightemp vs Lc0_maxtemp)
[1] Finished game 1 (Lc0_hightemp vs Lc0_maxtemp): 0-1 {time forfeit}
Score of Lc0_hightemp vs Lc0_maxtemp: 0 - 1 - 0  [0.000] 1
[1] Started game 2 of 200 (Lc0_hightemp vs Lc0_maxtemp)
[1] Finished game 2 (Lc0_hightemp vs Lc0_maxtemp): 1/2-1/2 {3-fold repetition}
Score of Lc0_hightemp vs Lc0_maxtemp: 0 - 1 - 1  [0.250] 2
[1] Started game 3 of 200 (Lc0_hightemp vs Lc0_maxtemp)
[1] Finished game 3 (Lc0_hightemp vs Lc0_maxtemp): 1/2-1/2 {3-fold repetition}
Score of Lc0_hightemp vs Lc0_maxtemp: 0 - 1 - 2  [0.333] 3
[1] Started game 4 of 200 (Lc0_hightemp vs Lc0_maxtemp)
[1] Finished game 4 (Lc0_hightemp vs Lc0_maxtemp): 1/2-1/2 {3-fold repetition}
Score of Lc0_hightemp vs Lc0_maxtemp: 0 - 1 - 3  [0.375] 4
[1] Started game 5 of 200 (Lc0_hightemp vs Lc0_maxtemp)
[1] Finished game 5 (Lc0_hightemp vs Lc0_maxtemp): 1/2-1/2 {3-fold repetition}
Score of Lc0_hightemp vs Lc0_maxtemp: 0 - 1 - 4  [0.400] 5
[1] Started game 6 of 200 (Lc0_hightemp

       _       _
|   _ | |
|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026

|   _ | |
|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026
Search algorithm: classicSearch algorithm: classic

       _
|   _ | |
|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026       _

|   _ | |
|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026
Search algorithm: classic
Search algorithm: classic
Loading weights file from: /kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz
Loading weights file from: /kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz
Weights file has multihead format, updating format flag
Weights file has multihead format, updating format flag
CUDA Runtime version: 12.8.0
CUDA Runtime version: 12.8.0
Latest version of CUDA supported by the driver: 13.0.0
GPU: Tesla T4
GPU memory: 14.5622 Gb
GPU clock frequency: 1590 MHz
GPU compute capability: 7.5
L2 cache capacity: 4194304
Latest version of CUDA supported by the driver: 13.0.0
GPU: Tesla T4
GPU 

In [ ]:
%%bash
chmod +x /kaggle/working/lc0_gpu0.sh /kaggle/working/lc0_gpu1.sh
chmod +x /kaggle/working/crackfish1924

RUY_LOPEZ_FEN="rn2kbr1/ppp1pp1p/3q1npB/3p4/3P4/3Q1NPb/PPP1PP1P/RN2KBR1 w Qq - 2 7"
echo "${RUY_LOPEZ_FEN}" > /kaggle/working/ruy_lopez_start.epd

/kaggle/working/c-chess-cli/c-chess-cli \
  -each tc=20/1 \
  -engine name=Lc0_hightemp cmd=/kaggle/working/crackfish1924 \
    option.Backend=cuda-fp16 \
    "option.WeightsFile=/kaggle/working/lc0/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz" \
    option.Cpuct=20 \
    option.PolicyTemperature=4 \
    option.SmartPruningFactor=1.0 \
    option.CPuctFactor=1 \
  -engine name=Mad_stockfish cmd=/kaggle/working/crackfish1924 \
    option.Hash=4096 \
    option.Threads=4 \
  -openings file=/kaggle/working/ruy_lopez_start.epd \
  -repeat \
  -rounds 2 \
  -pgn /kaggle/working/lc0basetrain.pgn

In [ ]:
!tail -n 100 /kaggle/working/lc0basetrain.pgn

In [7]:
!pip -q install python-chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 52.5 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done


In [8]:
import chess
import chess.pgn
import random

PGN = "/kaggle/working/lc0basetrain.pgn"
TRAIN_TXT = "/kaggle/working/cf_train.txt"
VAL_TXT = "/kaggle/working/cf_val.txt"

PLY_STRIDE = 1
MIN_GAME_PLY = 4
SKIP_LAST_PLIES = 2
VAL_FRACTION = 0.10
SEED = 1924

random.seed(SEED)

games = []

with open(PGN, encoding="utf-8-sig") as f:
    while True:
        game = chess.pgn.read_game(f)

        if game is None:
            break

        games.append(game)

print(f"Loaded {len(games)} games")

indices = list(range(len(games)))
random.shuffle(indices)

n_val_games = max(1, round(len(indices) * VAL_FRACTION))
val_games = set(indices[:n_val_games])

def result_value(game):
    result = game.headers.get("Result", "*")

    if result == "1-0":
        return 1
    elif result == "0-1":
        return -1
    elif result == "1/2-1/2":
        return 0
    else:
        return 0

def extract(game, game_id):
    board = game.board()
    moves = list(game.mainline_moves())
    result = result_value(game)

    records = []

    for ply, move in enumerate(moves):

        if (
            MIN_GAME_PLY <= ply
            and ply < len(moves) - SKIP_LAST_PLIES
            and ply % PLY_STRIDE == 0
        ):
            if board.turn == chess.WHITE:
                pov_result = result
            else:
                pov_result = -result

            records.append(
                (
                    board.fen(),
                    move.uci(),
                    ply,
                    game_id,
                    pov_result
                )
            )

        board.push(move)

    return records

train_raw = []
val_raw = []

for i, game in enumerate(games):
    records = extract(game, i)

    if i in val_games:
        val_raw.extend(records)
    else:
        train_raw.extend(records)

def position_key(fen):
    return " ".join(fen.split()[:4])

def dedupe(records, forbidden=None):
    forbidden = forbidden or set()

    seen = set()
    out = []

    for record in records:
        k = position_key(record[0])

        if k not in seen and k not in forbidden:
            seen.add(k)
            out.append(record)

    return out, seen

train, train_keys = dedupe(train_raw)
val, val_keys = dedupe(val_raw, train_keys)

print(f"Raw positions: {len(train_raw)} train / {len(val_raw)} val")
print(f"Unique positions: {len(train)} train / {len(val)} val")

def write_records(records, path):
    written = 0

    with open(path, "w") as f:
        for fen, move, ply, game_id, result in records:

            board = chess.Board(fen)
            move_obj = chess.Move.from_uci(move)

            if move_obj not in board.legal_moves:
                continue

            cp = result * 1000

            f.write(
                f"{fen}|{move}|{cp}|{ply}|{result}\n"
            )

            written += 1

    return written

n_train = write_records(train, TRAIN_TXT)
n_val = write_records(val, VAL_TXT)

print(f"Games: {len(games)}")
print(f"Train records: {n_train}")
print(f"Validation records: {n_val}")
print(f"Train: {TRAIN_TXT}")
print(f"Validation: {VAL_TXT}")

Loaded 2509 games
Raw positions: 209005 train / 22271 val
Unique positions: 26989 train / 799 val
Games: 2509
Train records: 26989
Validation records: 799
Train: /kaggle/working/cf_train.txt
Validation: /kaggle/working/cf_val.txt


In [9]:
!/kaggle/working/nnue-pytorch/txt2binpack \
  /kaggle/working/cf_train.txt \
  /kaggle/working/cf_train.binpack

!/kaggle/working/nnue-pytorch/txt2binpack \
  /kaggle/working/cf_val.txt \
  /kaggle/working/cf_val.binpack

!ls -lh /kaggle/working/cf_train.binpack /kaggle/working/cf_val.binpack

Written:  26989
Rejected: 0
Written:  799
Rejected: 0
-rw-r--r-- 1 root root  64K Oct 10 06:11 /kaggle/working/cf_train.binpack
-rw-r--r-- 1 root root 3.2K Oct 10 06:11 /kaggle/working/cf_val.binpack


In [4]:
%cd /kaggle/working/nnue-pytorch

!bash compile_data_loader.bat

/kaggle/working/nnue-pytorch
ROOT_DIR: /kaggle/working/nnue-pytorch
SRC_DIR: data_loader/cpp/
BUILD_DIR: build
PGO_DIR: pgo_data
PGO_INPUT: /kaggle/working/nnue-pytorch/.pgo/small.binpack
Cleaning previous build and profile data...
Configuring PGO_Generate build...
-- The CXX compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Adding BMI2 support
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Configuring done (1.2s)
-- Generating done (0.0s)
-- Build files have been written to: /kaggle/working/nnue-pytorch/build
Building instrumented default targets...
[ 11%] Building CXX object CMakeFiles/training_data_loader_report.dir/training_data_loader.cpp.o
[ 22%] Building CXX object CMakeFiles/training_data_loader_report.dir/

In [10]:
%cd /kaggle/working/nnue-pytorch

!find build -name "libtraining_data_loader.so" -print
!python train.py --help

/kaggle/working/nnue-pytorch
build/libtraining_data_loader.so
Traceback (most recent call last):
  File "/kaggle/working/nnue-pytorch/train.py", line 9, in <module>
    import tyro
ModuleNotFoundError: No module named 'tyro'


In [11]:
%cd /kaggle/working/nnue-pytorch
!pip install tyro
!python train.py \
    /kaggle/working/cf_train.binpack \
    --validation-datasets /kaggle/working/cf_val.binpack \
    --validation-size 799 \
    --check-val-every-n-epoch 1 \
    --accelerator cuda \
    --gpus 0,1 \
    --batch-size 256 \
    --epoch-size 26989 \
    --max-epochs 320 \
    --network-save-period 160 \
    --save-last-network True \
    --default-root-dir /kaggle/working/crackedfish_nnue \
    --lambda 0.0 \
    --warmup-steps 100 \
    --no-filtered \
    --no-wld-filtered \
    --soft-early-fen-skipping 0

/kaggle/working/nnue-pytorch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 3.6 MB/s eta 0:00:0000:01
batch_size(global)=256 | n_devices=2 | batch_size(per_gpu)=128
Loss parameters:
LossParams(in_offset=270, out_offset=270, in_scaling=340, out_scaling=380, lambda_config=LambdaConfig(start_lambda=1.0, end_lambda=1.0, jitter_lambda_sample=0.0, jitter_lambda_batch=0.0, jitter_decay_lambda_batch=0.0, lambda_=0.0, lambda_schedule_steps=-1, lambda_cycle_warmup_pct=0.3, lambda_cycle_delta=0.0, lambda_cycle_jitter=False), pow_exp=2.5, qp_asymmetry=0.0, w1=0.0, w2=0.5, tb_remap_base=4000.0, tb_remap_scale=4000.0, tb_remap_decay=0.85)
Feature set: Full_Threats+PP_3Wide+HalfKAv2_hm^
Num inputs: 88944
Training with: ('/kaggle/working/cf_train.binpack',)
Validating with: ('/kaggle/working/cf_val.binpack',)
Seed 42
DataloaderSkipConfig(filtered=False, wld_filtered=False, random_fen_skipping=0, early_fen_skipping=-1, soft_early_fen_skipping=0, simple_eval_skipping=-1, param_index=0, pc_y0

In [12]:
!find /kaggle/working/crackedfish_nnue -type f \
  -printf '%s %p\n' | sort -n

0 /kaggle/working/crackedfish_nnue/training_finished
124818 /kaggle/working/crackedfish_nnue/lightning_logs/version_0/metrics.csv
194346 /kaggle/working/crackedfish_nnue/lightning_logs/version_0/events.out.tfevents.1791612794.27dd6b140625.364501.0
1857951813 /kaggle/working/crackedfish_nnue/lightning_logs/version_0/checkpoints/last.ckpt
1857956421 /kaggle/working/crackedfish_nnue/lightning_logs/version_0/checkpoints/epoch=159-step=33600.ckpt
1857956485 /kaggle/working/crackedfish_nnue/lightning_logs/version_0/checkpoints/epoch=319-step=67200.ckpt


In [13]:
%%bash
# Create a safe backup of your true last checkpoint in the root directory
cp /kaggle/working/crackedfish_nnue/lightning_logs/version_0/checkpoints/last.ckpt /kaggle/working/Crackfish1924_final.ckpt

# Purge the cluttered logging workspace entirely
rm -rf /kaggle/working/crackedfish_nnue

# Verify that only your clean master checkpoint remains
ls -lh /kaggle/working/


total 2.0G
drwxr-xr-x  6 root root 4.0K Oct 10 05:37 c-chess-cli
-rw-r--r--  1 root root  64K Oct 10 06:11 cf_train.binpack
-rw-r--r--  1 root root 1.6M Oct 10 06:10 cf_train.txt
-rw-r--r--  1 root root 3.2K Oct 10 06:11 cf_val.binpack
-rw-r--r--  1 root root  44K Oct 10 06:10 cf_val.txt
-rw-r--r--  1 root root  87M Oct 10 05:37 crackedfish.nnue
-rw-r--r--  1 root root  95M Oct 10 05:37 crackfish1924
-rw-r--r--  1 root root 1.8G Oct 10 07:10 Crackfish1924_final.ckpt
-rw-r--r--  1 root root 6.1K Oct 10 05:37 isitbetteronmoreram.pgn
drwxr-xr-x 14 root root 4.0K Oct 10 05:37 lc0
-rw-r--r--  1 root root 5.0M Oct 10 06:10 lc0basetrain.pgn
-rwxr-xr-x  1 root root   90 Oct 10 05:42 lc0_gpu0.sh
-rwxr-xr-x  1 root root   90 Oct 10 05:42 lc0_gpu1.sh
-rw-r--r--  1 root root  18K Oct 10 05:37 madness.pgn
drwxr-xr-x 14 root root 4.0K Oct 10 05:37 nnue-pytorch
-rw-r--r--  1 root root 1018 Oct 10 05:42 process1.sh
-rw-r--r--  1 root root 1018 Oct 10 05:42 process2.sh
-rw-r--r--  1 root root   65 Oct 

In [14]:
import torch

p = "/kaggle/working/Crackfish1924_final.ckpt"

ckpt = torch.load(p, map_location="cpu", weights_only=False)

print("Checkpoint loaded successfully")
print("epoch:", ckpt.get("epoch"))
print("global_step:", ckpt.get("global_step"))
print("keys:", ckpt.keys())

Checkpoint loaded successfully
epoch: 319
global_step: 67200
keys: dict_keys(['epoch', 'global_step', 'state_dict', 'lr_schedulers', 'jitter_buffer_value'])


In [16]:
import csv

p = "/kaggle/working/crackedfish_nnue/lightning_logs/version_0/metrics.csv"

with open(p, newline="") as f:
    rows = list(csv.DictReader(f))

print("Columns:", rows[0].keys())
print("Rows:", len(rows))
print()

for row in rows:
    print(row)


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/crackedfish_nnue/lightning_logs/version_0/metrics.csv'

In [18]:
!du -sh /kaggle/working/crackedfish_nnue/lightning_logs/version_0/checkpoints
!ls -lh /kaggle/working/crackedfish_nnue/lightning_logs/version_0/checkpoints

du: cannot access '/kaggle/working/crackedfish_nnue/lightning_logs/version_0/checkpoints': No such file or directory
ls: cannot access '/kaggle/working/crackedfish_nnue/lightning_logs/version_0/checkpoints': No such file or directory


In [ ]:
!chmod +x /kaggle/working/crackfish1924

In [19]:
%cd /kaggle/working/nnue-pytorch

!python serialize.py \
  "/kaggle/working/Crackfish1924_final.ckpt" \
  "/kaggle/working/crackedfish.nnue" \
  --description "Crackedfish cursed NNUE"
!ls -lh /kaggle/working/crackedfish.nnue
!sha256sum /kaggle/working/crackedfish.nnue
!chmod +x /kaggle/working/crackfish1924
!printf 'uci\nsetoption name EvalFile value /kaggle/working/crackedfish.nnue\nisready\nposition startpos\ngo depth 1\nquit\n' | /kaggle/working/crackfish1924 2>&1

/kaggle/working/nnue-pytorch
Converting /kaggle/working/Crackfish1924_final.ckpt to /kaggle/working/crackedfish.nnue
---------------
ft_bias: 
-4        | ##########
2         | ############
9         | ##################################################
16        | ########
23        | #
30        | 
37        | 
Number of elements: 1024
Minimum value in layer is -4, occurring 1 times.
Maximum value in layer is 43, occurring 1 times.
---------------
---------------
Full_Threats ft_weight: 
-28       | 
-18       | 
-9        | #####################
0         | ##################################################
8         | 
17        | 
26        | 
Number of elements: 61243392
Minimum value in layer is -28, occurring 2 times.
Maximum value in layer is 35, occurring 1 times.
---------------
---------------
Full_Threats psqt_weight: 
-1874     | 
-1332     | 
-790      | 
-248      | ##################################################
292       | 
834       | 
1376      | 
Number of eleme

In [21]:
%%bash
printf 'uci\nquit\n' | /kaggle/working/stockfish/stockfish-linux-x86-64-universal

# Lc0: print every UCI option
printf 'uci\nquit\n' | /kaggle/working/lc0/build/release/lc0

id name Lc0 v0.34.0-dev+git.043e723
id author The LCZero Authors.
option name LogFile type string default 
option name ConfigFile type string default lc0.config
option name Ponder type check default false
option name SyzygyPath type string default 
option name Threads type spin default 0 min 0 max 128
option name MinibatchSize type spin default 0 min 0 max 1024
option name CPuct type string default 1.745000
option name CPuctBase type string default 38739.000000
option name CPuctFactor type string default 3.894000
option name TwoFoldDraws type check default true
option name VerboseMoveStats type check default false
option name FpuStrategy type combo default reduction var reduction var absolute
option name FpuValue type string default 0.330000
option name CacheHistoryLength type spin default 0 min 0 max 7
option name MaxCollisionEvents type spin default 917 min 1 max 65536
option name MaxCollisionVisits type spin default 80000 min 1 max 100000000
option name MaxCollisionVisitsScalingStar

bash: line 1: /kaggle/working/stockfish/stockfish-linux-x86-64-universal: Permission denied
       _
|   _ | |
|_ |_ |_| v0.34.0-dev+git.043e723 built Sep 16 2026
Search algorithm: classic


In [ ]:
!sed -i 's/optimism\[us\]  = 114 \* avg/optimism[us]  = 1924 * avg/' \
    /kaggle/working/stockfish/src/search.cpp

In [ ]:
!sed -i \
's/Depth R = 7 + depth \/ 3 + std::max((ss->staticEval - beta) \/ 256, 0);/Depth R = 1 + depth \/ 64 + std::max((ss->staticEval - beta) \/ 919191, 0);/' \
/kaggle/working/stockfish/src/search.cpp

!grep -n -A2 -B2 "Depth R =" /kaggle/working/stockfish/src/search.cpp

In [ ]:
%%bash
cd /kaggle/working/stockfish/src
make clean
cp /kaggle/working/crackedfish.nnue /kaggle/working/stockfish/src/nn.nnue
make build ARCH=x86-64-universal EvalFile=/kaggle/working/stockfish/src/nn.nnue -j$(nproc)
chmod +x /kaggle/working/crackfish1924

In [ ]:
!grep -n -E \
"reduction|Reduction|nullMove|NullMove|singular|Singular|aspiration|delta|correctionHistory|Correction|optimumTime|maximumTime" \
/kaggle/working/stockfish/src/search.cpp \
/kaggle/working/stockfish/src/timeman.cpp | head -150

In [ ]:
!sed -i \
's/reductions\[i\] = int(2872 \/ 128.0 \* std::log(i));/reductions[i] = int(14 \/ 1280.0 * std::log(i));/' \
/kaggle/working/stockfish/src/search.cpp

!grep -n "reductions\[i\]" /kaggle/working/stockfish/src/search.cpp

In [ ]:
!cd /kaggle/working && git clone https://github.com/official-stockfish/nnue-pytorch.git
!cd nnue-pytorch

In [ ]:
import subprocess

SF = "/kaggle/working/stockfish/stockfish-linux-x86-64-universal"

p = subprocess.run(
    [SF],
    input="uci\nquit\n",
    text=True,
    capture_output=True
)

print(p.stdout)

In [ ]:
%%bash
cd /kaggle/working/nnue-pytorch
#find . -maxdepth 3 -type f | sort | head -150
ls -lah data_loader
find data_loader -maxdepth 3 -type f | sort

In [ ]:
!grep -nE "fromFen|fromFEN|setFen|setFEN|fen|FEN|fromUci|fromUCI|uci|UCI" \
/kaggle/working/nnue-pytorch/data_loader/cpp/lib/chess.h | head -100

In [ ]:
from pathlib import Path

code = r'''
#include <iostream>
#include <fstream>
#include <sstream>
#include <string>
#include <cstdint>

#include "data_loader/cpp/lib/binpack.h"
#include "data_loader/cpp/lib/chess.h"
#include "data_loader/cpp/lib/training_data_entry.h"

static std::string trim(std::string s)
{
    const auto first = s.find_first_not_of(" \t\r\n");
    if (first == std::string::npos)
        return "";

    const auto last = s.find_last_not_of(" \t\r\n");
    return s.substr(first, last - first + 1);
}

int main(int argc, char** argv)
{
    if (argc != 3)
    {
        std::cerr
            << "Usage: txt2binpack INPUT.txt OUTPUT.binpack\n";
        return 1;
    }

    std::ifstream input(argv[1]);

    if (!input)
    {
        std::cerr << "Could not open input file.\n";
        return 1;
    }

    // Truncate so rerunning the Kaggle cell doesn't append
    // another copy to an existing dataset.
    binpack::CompressedTrainingDataEntryWriter writer(
        argv[2],
        std::ios_base::trunc
    );

    std::string line;
    std::size_t lineNumber = 0;
    std::size_t written = 0;
    std::size_t rejected = 0;

    while (std::getline(input, line))
    {
        ++lineNumber;

        if (line.empty())
            continue;

        // Format:
        // FEN|MOVE|SCORE|PLY|RESULT

        std::stringstream ss(line);

        std::string fen;
        std::string moveText;
        std::string scoreText;
        std::string plyText;
        std::string resultText;

        if (!std::getline(ss, fen, '|') ||
            !std::getline(ss, moveText, '|') ||
            !std::getline(ss, scoreText, '|') ||
            !std::getline(ss, plyText, '|') ||
            !std::getline(ss, resultText))
        {
            std::cerr
                << "Malformed line "
                << lineNumber
                << "\n";

            ++rejected;
            continue;
        }

        fen        = trim(fen);
        moveText   = trim(moveText);
        scoreText  = trim(scoreText);
        plyText    = trim(plyText);
        resultText = trim(resultText);

        try
        {
            auto posOptional =
                chess::Position::tryFromFen(fen);

            if (!posOptional)
            {
                std::cerr
                    << "Invalid FEN line "
                    << lineNumber
                    << "\n";

                ++rejected;
                continue;
            }

            chess::Position pos = *posOptional;

            chess::Move move =
                chess::uci::uciToMove(pos, moveText);

            binpack::TrainingDataEntry entry;

            entry.pos    = pos;
            entry.move   = move;
            entry.score  =
                static_cast<std::int16_t>(
                    std::stoi(scoreText)
                );

            entry.ply =
                static_cast<std::uint16_t>(
                    std::stoul(plyText)
                );

            entry.result =
                static_cast<std::int16_t>(
                    std::stoi(resultText)
                );

            if (!entry.isValid())
            {
                std::cerr
                    << "Illegal move on line "
                    << lineNumber
                    << ": "
                    << moveText
                    << "\n";

                ++rejected;
                continue;
            }

            writer.addTrainingDataEntry(entry);
            ++written;
        }
        catch (const std::exception& e)
        {
            std::cerr
                << "Line "
                << lineNumber
                << ": "
                << e.what()
                << "\n";

            ++rejected;
        }
    }

    std::cout
        << "Written:  "
        << written
        << "\nRejected: "
        << rejected
        << "\n";

    return rejected == 0 ? 0 : 2;
}
'''

Path("/kaggle/working/nnue-pytorch/txt2binpack.cpp").write_text(code)

print("Created txt2binpack.cpp")

In [ ]:
%cd /kaggle/working/nnue-pytorch

!g++ -std=c++20 -O3 \
    -I. \
    txt2binpack.cpp \
    -o txt2binpack

In [ ]:
!rm -f /kaggle/working/test.binpack

!/kaggle/working/nnue-pytorch/txt2binpack \
    /kaggle/working/test_training.txt \
    /kaggle/working/test.binpack

!ls -lh /kaggle/working/test.binpack
!xxd -l 64 /kaggle/working/test.binpack

In [ ]:
#!pip install chess
import chess
import chess.engine
import random

def stockfish_randomized_opening(
    engine,
    min_plies=6,
    max_plies=12,
    nodes=2000,
    multipv=5
):
    board = chess.Board()

    opening_length = random.randint(min_plies, max_plies)

    for _ in range(opening_length):

        if board.is_game_over():
            break

        infos = engine.analyse(
            board,
            chess.engine.Limit(nodes=nodes),
            multipv=multipv
        )

        candidates = []

        for info in infos:
            pv = info.get("pv")

            if pv:
                candidates.append(pv[0])

        if not candidates:
            break

        move = random.choice(candidates)

        board.push(move)

    return board

In [ ]:
SF = "/kaggle/working/stockfish/stockfish-linux-x86-64-universal"

engine = chess.engine.SimpleEngine.popen_uci(SF)

board = stockfish_randomized_opening(
    engine,
    min_plies=6,
    max_plies=12,
    nodes=2000,
    multipv=5
)

print(board)
print()
print("FEN:")
print(board.fen())

engine.quit()

In [ ]:
import os
from IPython.display import FileLink

# Zip your modified Stockfish source directory directly
os.system("zip -q -r /kaggle/working/Mac_tuned_sf19.zip /kaggle/working/stockfish")

# Print the download link
FileLink(r'Mac_tuned_sf19.zip')
'''
cd ~/Downloads
unzip -q Mac_tuned_sf19.zip
cd ~/Downloads/stockfish/src

# Clean old artifacts and compile natively for Mac Silicon
make clean
make profile-build ARCH=apple-silicon COMP=clang EvalFile=crackfish1924.nnue -j$(sysctl -n hw.ncpu)

# Apply Apple's free local memory signature to whitelist the binary
codesign --force --deep -s - ./stockfish
mv stockfish ~/Downloads/Crackfish1924_Mac


xattr -d com.apple.quarantine ~/Downloads/Mac_tuned_sf19 && chmod +x ~/Downloads/Mac_tuned_sf19

Run this on mac
'''

In [ ]:
!rm -rf /kaggle/working/Mac_tuned_sf19.zip

# 